In [ ]:
import pandas as pd


In [ ]:
fake_df = pd.read_csv("../data/raw/Fake.csv")


In [ ]:
true_df = pd.read_csv("../data/raw/True.csv")


In [ ]:
print(fake_df.shape)
print(true_df.shape)

fake_df.head()


In [ ]:
print(fake_df.columns)
print(true_df.columns)


In [ ]:
fake_df["label"] = 1
true_df["label"] = 0


In [ ]:
fake_df.head()


In [ ]:
import pandas as pd

df = pd.concat([fake_df, true_df], axis=0)


In [ ]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
print(df.shape)
df.head()


In [ ]:
df.columns


In [ ]:
import re


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text


In [ ]:
df["clean_text"] = df["text"].apply(clean_text)


In [ ]:
df[["text", "clean_text"]].head()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Split raw text before fitting the vectorizer. This keeps vocabulary
# discovery confined to the training partition.
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)


In [ ]:
print(X.shape)
print(y.shape)


In [ ]:
# The train/test split and vectorizer fitting now happen together above.


In [ ]:
from sklearn.linear_model import LogisticRegression


In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)


In [ ]:
import joblib


In [ ]:
import joblib

joblib.dump(model, "../models/fake_news_model.pkl")


In [ ]:
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")


In [ ]:
def describe_model_output(prediction):
    label = "dataset-labeled fake" if int(prediction) == 1 else "dataset-labeled real"
    return (
        f"The classifier assigned the {label} label based on text patterns learned "
        "from the historical training dataset. This output does not verify claims, "
        "check cited sources, or establish whether an article is true or false."
    )


In [ ]:
sample_text = X_test_text.iloc[0]
sample_vector = vectorizer.transform([sample_text])
sample_pred = model.predict(sample_vector)[0]

print("Model-assigned label:", "dataset-labeled fake" if sample_pred == 1 else "dataset-labeled real")
print("Interpretation:", describe_model_output(sample_pred))
